# Physics Solution Class Feature Selection v1.0

`tools/physics_solution` 기반 feature selection notebook입니다.

목표:
- image handcrafted feature 추출
- geometry feature 및 motion feature 결합
- class와 feature 간 correlation 분석
- classification 기반 feature importance 추출
- interaction / PCA / clustering 기반 상위 feature 조합 후보 탐색


In [1]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, normalized_mutual_info_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / 'tools' / 'physics_solution').exists() and (ROOT.parent / 'tools' / 'physics_solution').exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / 'tools' / 'physics_solution'))
from geometry_reasoning import GeometryReasoningConfig, extract_geometry_features

DATA_ROOT = ROOT / 'data'
OUT_DIR = ROOT / 'outputs' / 'physics_feature_analysis'
MOTION_CSV = DATA_ROOT / 'motion_targets.csv'
INCLUDE_TEST_FEATURE_EXPORT = False
MIN_AUC_FOR_IMPORTANCE = 0.70
TOP_INTERACTION_FEATURES = 18
MAX_INTERACTIONS = 40
PCA_COMPONENTS = 5
CLUSTER_COUNT = 4
MAX_SAMPLES_PER_SPLIT = None  # smoke test는 10 같은 값으로 설정
LABEL_MAP = {'stable': 0, 'unstable': 1}

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('OUT_DIR:', OUT_DIR)


ROOT: /media/hdd0/whyz/structure-stability
DATA_ROOT: /media/hdd0/whyz/structure-stability/data
OUT_DIR: /media/hdd0/whyz/structure-stability/outputs/physics_feature_analysis


In [2]:
def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding='utf-8-sig')


def read_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def fft_high_freq_ratio(gray: np.ndarray, center_frac: float = 0.18) -> float:
    gray_f = gray.astype(np.float32)
    fft = np.fft.fftshift(np.fft.fft2(gray_f))
    mag = np.abs(fft)
    h, w = gray.shape
    cy, cx = h // 2, w // 2
    ry, rx = int(h * center_frac / 2), int(w * center_frac / 2)
    low_mask = np.zeros_like(gray_f, dtype=bool)
    low_mask[max(0, cy - ry): min(h, cy + ry + 1), max(0, cx - rx): min(w, cx + rx + 1)] = True
    return float(mag[~low_mask].sum() / (mag.sum() + 1e-6))


def hue_entropy(hue: np.ndarray, sat: np.ndarray, bins: int = 36) -> float:
    valid = sat > 25
    if int(valid.sum()) < 64:
        return float('nan')
    hist, _ = np.histogram(hue[valid], bins=bins, range=(0, 180), density=True)
    hist = hist[hist > 0]
    return float(-(hist * np.log(hist + 1e-9)).sum())


def estimate_structure_mask(rgb: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    h, w = lab.shape[:2]
    border = np.concatenate([
        lab[:18].reshape(-1, 3),
        lab[-18:].reshape(-1, 3),
        lab[:, :18].reshape(-1, 3),
        lab[:, -18:].reshape(-1, 3),
    ], axis=0)
    center = np.median(border, axis=0)
    dist = np.linalg.norm(lab.astype(np.float32) - center.astype(np.float32), axis=2)
    mask = (dist > np.percentile(dist, 84)).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    if n_labels <= 1:
        return np.zeros((h, w), dtype=np.uint8)

    image_center = np.array([w / 2, h / 2], dtype=np.float32)
    best_idx = None
    best_score = None
    for idx in range(1, n_labels):
        area = int(stats[idx, cv2.CC_STAT_AREA])
        if area < 80:
            continue
        x = int(stats[idx, cv2.CC_STAT_LEFT])
        y = int(stats[idx, cv2.CC_STAT_TOP])
        ww = int(stats[idx, cv2.CC_STAT_WIDTH])
        hh = int(stats[idx, cv2.CC_STAT_HEIGHT])
        centroid = np.array([x + ww / 2, y + hh / 2], dtype=np.float32)
        center_penalty = np.linalg.norm((centroid - image_center) / np.array([w, h], dtype=np.float32))
        score = float(area - 4000 * center_penalty)
        if best_score is None or score > best_score:
            best_score = score
            best_idx = idx

    out = np.zeros((h, w), dtype=np.uint8)
    if best_idx is not None:
        out[labels == best_idx] = 255
    return out


def detect_checker_lines(gray: np.ndarray, view: str) -> list[tuple[int, int, int, int, float, float]]:
    h, w = gray.shape
    if view == 'front':
        roi = gray[h // 4:, :]
        y_offset = h // 4
    else:
        roi = gray
        y_offset = 0

    edges = cv2.Canny(cv2.GaussianBlur(roi, (5, 5), 0), 60, 140)
    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=55,
        minLineLength=max(20, int(0.08 * w)),
        maxLineGap=8,
    )
    if lines is None:
        return []

    out = []
    for line in lines[:, 0, :]:
        x1, y1, x2, y2 = map(int, line)
        y1 += y_offset
        y2 += y_offset
        dx = x2 - x1
        dy = y2 - y1
        length = float(math.hypot(dx, dy))
        if length < max(20, 0.08 * w):
            continue
        angle = math.degrees(math.atan2(dy, dx))
        angle = ((angle + 90) % 180) - 90
        if view == 'front' and abs(angle) < 8:
            continue
        out.append((x1, y1, x2, y2, length, angle))
    out.sort(key=lambda x: x[4], reverse=True)
    return out[:40]


def line_to_abc(line: tuple[int, int, int, int, float, float]) -> np.ndarray:
    x1, y1, x2, y2, _, _ = line
    a = y1 - y2
    b = x2 - x1
    c = x1 * y2 - x2 * y1
    norm = math.hypot(a, b) + 1e-6
    return np.array([a / norm, b / norm, c / norm], dtype=np.float32)


def intersections_from_lines(lines: list[tuple[int, int, int, int, float, float]], width: int, height: int) -> tuple[float, float, float, int]:
    if len(lines) < 2:
        return float('nan'), float('nan'), float('nan'), 0
    points = []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            if abs(lines[i][5] - lines[j][5]) < 12:
                continue
            l1 = line_to_abc(lines[i])
            l2 = line_to_abc(lines[j])
            x = np.cross(l1, l2)
            if abs(x[2]) < 1e-6:
                continue
            px = x[0] / x[2]
            py = x[1] / x[2]
            if -width <= px <= 2 * width and -height <= py <= 2 * height:
                points.append((px, py))
    if len(points) < 3:
        return float('nan'), float('nan'), float('nan'), len(points)
    pts = np.asarray(points, dtype=np.float32)
    median = np.median(pts, axis=0)
    spread = np.median(np.linalg.norm(pts - median, axis=1)) / math.hypot(width, height)
    return float(median[0]), float(median[1]), float(spread), len(points)


In [3]:
def line_curvature_proxy(gray: np.ndarray, lines: list[tuple[int, int, int, int, float, float]], max_lines: int = 6) -> float:
    edges = cv2.Canny(gray, 60, 140)
    yy, xx = np.where(edges > 0)
    if len(xx) == 0 or len(lines) == 0:
        return float('nan')
    residuals = []
    for line in lines[:max_lines]:
        x1, y1, x2, y2, _, _ = line
        xmin, xmax = sorted([x1, x2])
        ymin, ymax = sorted([y1, y2])
        pad = 4
        mask = (xx >= xmin - pad) & (xx <= xmax + pad) & (yy >= ymin - pad) & (yy <= ymax + pad)
        if int(mask.sum()) < 25:
            continue
        pts = np.column_stack([xx[mask], yy[mask]]).astype(np.float32)
        p1 = np.array([x1, y1], dtype=np.float32)
        p2 = np.array([x2, y2], dtype=np.float32)
        v = p2 - p1
        dist = np.abs((pts[:, 0] - x1) * v[1] - (pts[:, 1] - y1) * v[0]) / (np.linalg.norm(v) + 1e-6)
        residuals.append(float(np.median(dist)))
    if not residuals:
        return float('nan')
    return float(np.median(residuals))


def estimate_shadow_metrics(rgb: np.ndarray, structure_mask: np.ndarray, view: str) -> tuple[float, float, float]:
    if view != 'front':
        return float('nan'), float('nan'), float('nan')
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h, w = hsv.shape[:2]
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    roi_mask[h // 3:, :] = 1
    dark_thr = np.percentile(val[roi_mask.astype(bool)], 28)
    shadow = ((sat < 70) & (val < dark_thr) & (roi_mask > 0)).astype(np.uint8)
    shadow[structure_mask > 0] = 0
    shadow = cv2.morphologyEx(shadow, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    shadow = cv2.morphologyEx(shadow, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(shadow, connectivity=8)
    best_idx = None
    best_area = 0
    for idx in range(1, n_labels):
        area = int(stats[idx, cv2.CC_STAT_AREA])
        y = int(stats[idx, cv2.CC_STAT_TOP])
        hh = int(stats[idx, cv2.CC_STAT_HEIGHT])
        if area < 120 or y + hh < h // 2:
            continue
        if area > best_area:
            best_area = area
            best_idx = idx
    if best_idx is None:
        return 0.0, float('nan'), float('nan')

    yy, xx = np.where(labels == best_idx)
    pts = np.column_stack([xx, yy]).astype(np.float32)
    centered = pts - pts.mean(axis=0, keepdims=True)
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    major = eigvecs[:, np.argmax(eigvals)]
    angle = math.degrees(math.atan2(major[1], major[0]))
    elongation = float(np.sqrt(eigvals.max() / (eigvals.min() + 1e-6)))
    return float(best_area / (h * w)), float(angle), elongation


def chessboard_detection(gray: np.ndarray, view: str) -> tuple[int, int, int]:
    if view != 'front':
        return 0, -1, -1
    flags = cv2.CALIB_CB_EXHAUSTIVE + cv2.CALIB_CB_ACCURACY
    for pattern in [(5, 5), (6, 6), (7, 7), (8, 8), (9, 9)]:
        ok, _ = cv2.findChessboardCornersSB(gray, pattern, flags=flags)
        if ok:
            return 1, pattern[0], pattern[1]
    return 0, -1, -1


def extract_image_features(split: str, sample_id: str, label: str | None, view: str, image_path: Path) -> dict:
    rgb = read_rgb(image_path)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h, w = gray.shape

    structure_mask = estimate_structure_mask(rgb)
    structure_area = float((structure_mask > 0).mean())
    ys, xs = np.where(structure_mask > 0)
    if len(xs):
        bbox_w = (xs.max() - xs.min() + 1) / w
        bbox_h = (ys.max() - ys.min() + 1) / h
        center_x = xs.mean() / w
        center_y = ys.mean() / h
    else:
        bbox_w = bbox_h = center_x = center_y = float('nan')

    lines = detect_checker_lines(gray, view)
    vp_x, vp_y, vp_spread, n_intersections = intersections_from_lines(lines, w, h)
    curvature = line_curvature_proxy(gray, lines)
    shadow_area, shadow_angle, shadow_elongation = estimate_shadow_metrics(rgb, structure_mask, view)
    board_found, board_nx, board_ny = chessboard_detection(gray, view)

    gray_f = gray.astype(np.float32)
    residual = gray_f - cv2.GaussianBlur(gray_f, (0, 0), 1.0)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]
    hue = hsv[:, :, 0]

    return {
        'split': split,
        'sample_id': sample_id,
        'label': label,
        'view': view,
        'image_path': str(image_path),
        'width': w,
        'height': h,
        'mean_r': float(rgb[:, :, 0].mean()),
        'mean_g': float(rgb[:, :, 1].mean()),
        'mean_b': float(rgb[:, :, 2].mean()),
        'std_r': float(rgb[:, :, 0].std()),
        'std_g': float(rgb[:, :, 1].std()),
        'std_b': float(rgb[:, :, 2].std()),
        'brightness_mean': float(val.mean() / 255.0),
        'brightness_std': float(val.std() / 255.0),
        'saturation_mean': float(sat.mean() / 255.0),
        'saturation_std': float(sat.std() / 255.0),
        'hue_entropy': hue_entropy(hue, sat),
        'laplacian_var': float(cv2.Laplacian(gray_f, cv2.CV_32F).var()),
        'noise_residual_std': float(residual.std()),
        'fft_high_freq_ratio': fft_high_freq_ratio(gray),
        'edge_density': float((cv2.Canny(gray, 60, 140) > 0).mean()),
        'structure_area_ratio': structure_area,
        'structure_bbox_w': float(bbox_w),
        'structure_bbox_h': float(bbox_h),
        'structure_center_x': float(center_x),
        'structure_center_y': float(center_y),
        'checker_line_count': int(len(lines)),
        'vp_x': vp_x,
        'vp_y': vp_y,
        'vp_spread': vp_spread,
        'vp_pitch_proxy': float(vp_y / h) if not np.isnan(vp_y) else float('nan'),
        'vp_intersections': int(n_intersections),
        'distortion_proxy': float(curvature / math.hypot(w, h)) if not np.isnan(curvature) else float('nan'),
        'shadow_area_ratio': shadow_area,
        'shadow_angle_deg': shadow_angle,
        'shadow_elongation': shadow_elongation,
        'chessboard_found': int(board_found),
        'chessboard_nx': int(board_nx),
        'chessboard_ny': int(board_ny),
    }


In [4]:
def build_labeled_catalog(data_root: Path, include_test: bool, max_samples_per_split: int | None) -> pd.DataFrame:
    rows = []
    for split in ['train', 'dev'] + (['test'] if include_test else []):
        if split == 'test':
            df = read_csv(data_root / 'sample_submission.csv')
            df['label'] = None
            sample_ids = df['id'].tolist()
            labels = [None] * len(sample_ids)
        else:
            df = read_csv(data_root / f'{split}.csv')
            sample_ids = df['id'].tolist()
            labels = df['label'].tolist()
        if max_samples_per_split is not None:
            sample_ids = sample_ids[:max_samples_per_split]
            labels = labels[:max_samples_per_split]
        for sample_id, label in zip(sample_ids, labels):
            for view in ['front', 'top']:
                image_path = data_root / split / sample_id / f'{view}.png'
                if image_path.exists():
                    rows.append({
                        'split': split,
                        'sample_id': sample_id,
                        'label': label,
                        'view': view,
                        'image_path': image_path,
                    })
    return pd.DataFrame(rows)


def build_image_feature_df(catalog_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in catalog_df.itertuples(index=False):
        rows.append(extract_image_features(row.split, row.sample_id, row.label, row.view, Path(row.image_path)))
    return pd.DataFrame(rows)


def build_geometry_df(data_root: Path, catalog_df: pd.DataFrame) -> pd.DataFrame:
    sample_meta = catalog_df[['split', 'sample_id', 'label']].drop_duplicates().reset_index(drop=True)
    rows = []
    cfg = GeometryReasoningConfig()
    for row in sample_meta.itertuples(index=False):
        front_path = data_root / row.split / row.sample_id / 'front.png'
        top_path = data_root / row.split / row.sample_id / 'top.png'
        if not front_path.exists() or not top_path.exists():
            continue
        front_rgb = read_rgb(front_path)
        top_rgb = read_rgb(top_path)
        feat = extract_geometry_features(front_rgb, top_rgb, cfg)
        feat['collapse_margin_proxy'] = float(np.clip(
            0.5 + 0.35 * (
                1.20 * feat['top_support_width_frac']
                + 0.90 * feat['front_base_width_frac']
                + 0.50 * feat['top_fill_ratio']
                - 0.75 * abs(feat['top_centroid_dx'])
                - 0.55 * abs(feat['front_tilt'])
                - 0.20 * feat['front_slenderness']
                - 0.25 * feat['front_top_heaviness']
            ),
            0.0,
            1.0,
        ))
        feat.update({'split': row.split, 'sample_id': row.sample_id, 'label': row.label})
        rows.append(feat)
    return pd.DataFrame(rows)


def load_motion_features(data_root: Path, motion_csv: Path | None) -> pd.DataFrame | None:
    motion_path = motion_csv if motion_csv is not None else data_root / 'motion_targets.csv'
    if not motion_path.exists():
        return None
    motion_df = read_csv(motion_path)
    keep_cols = [
        'id', 'max_diff_first', 'mean_diff_first', 'max_diff_prev', 'mean_diff_prev',
        'first_move_thr2', 'first_move_thr5', 'first_move_thr10',
        'severity_bucket', 'onset_bucket', 'soft_target',
    ]
    keep_cols = [c for c in keep_cols if c in motion_df.columns]
    return motion_df[keep_cols].rename(columns={'id': 'sample_id'})


def build_sample_level_features(image_feat_df: pd.DataFrame) -> pd.DataFrame:
    meta_cols = ['split', 'sample_id', 'label']
    value_cols = [c for c in image_feat_df.columns if c not in meta_cols + ['view', 'image_path']]

    wide_parts = []
    for view in sorted(image_feat_df['view'].unique()):
        part = image_feat_df.loc[image_feat_df['view'] == view, meta_cols + value_cols].copy()
        part = part.rename(columns={col: f'{view}_{col}' for col in value_cols})
        wide_parts.append(part)

    sample_level = wide_parts[0]
    for part in wide_parts[1:]:
        sample_level = sample_level.merge(part, on=meta_cols, how='outer')

    common_cols = [col for col in value_cols if f'front_{col}' in sample_level.columns and f'top_{col}' in sample_level.columns]
    derived_cols = {}
    for col in common_cols:
        delta = sample_level[f'front_{col}'] - sample_level[f'top_{col}']
        derived_cols[f'delta_{col}'] = delta
        derived_cols[f'abs_delta_{col}'] = delta.abs()
        derived_cols[f'mean_{col}'] = (sample_level[f'front_{col}'] + sample_level[f'top_{col}']) / 2.0

    if derived_cols:
        sample_level = pd.concat([sample_level, pd.DataFrame(derived_cols, index=sample_level.index)], axis=1)
    return sample_level.copy()


def build_analysis_df(image_feat_df: pd.DataFrame, geometry_df: pd.DataFrame, motion_df: pd.DataFrame | None) -> pd.DataFrame:
    sample_df = build_sample_level_features(image_feat_df)
    merged = sample_df.merge(geometry_df.drop(columns=['split', 'label'], errors='ignore'), on='sample_id', how='left')
    if motion_df is not None:
        merged = merged.merge(motion_df, on='sample_id', how='left')
    merged['target'] = merged['label'].map(LABEL_MAP)
    return merged


In [5]:
def get_feature_cols(df: pd.DataFrame) -> list[str]:
    excluded = {'split', 'sample_id', 'label', 'target'}
    return sorted([c for c in df.columns if c not in excluded and df[c].notna().any()])


def compute_correlation_outputs(df: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    corr_df = df[feature_cols + ['target']].copy().apply(pd.to_numeric, errors='coerce')
    matrix = corr_df.corr(method='pearson')
    target_corr = (
        matrix['target']
        .drop(labels=['target'])
        .sort_values(key=lambda s: s.abs(), ascending=False)
        .rename('corr_with_target')
        .to_frame()
    )
    target_corr['abs_corr_with_target'] = target_corr['corr_with_target'].abs()
    return matrix, target_corr


def build_models(feature_cols: list[str]) -> dict[str, Pipeline]:
    scaled_prep = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), feature_cols)
    ])
    tree_prep = ColumnTransformer([('num', SimpleImputer(strategy='median'), feature_cols)])
    return {
        'logreg': Pipeline([('prep', scaled_prep), ('clf', LogisticRegression(max_iter=4000, random_state=42))]),
        'rf': Pipeline([('prep', tree_prep), ('clf', RandomForestClassifier(n_estimators=600, min_samples_leaf=2, random_state=42, n_jobs=-1))]),
        'extra_trees': Pipeline([('prep', tree_prep), ('clf', ExtraTreesClassifier(n_estimators=800, min_samples_leaf=2, random_state=42, n_jobs=-1))]),
    }


def cross_validated_classification(df: pd.DataFrame, feature_cols: list[str], min_auc_for_importance: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    work_df = df[df['target'].notna()].copy().reset_index(drop=True)
    X = work_df[feature_cols]
    y = work_df['target'].astype(int).to_numpy()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    eval_rows = []
    importance_rows = []
    for model_name, model in build_models(feature_cols).items():
        oof_pred = np.zeros(len(work_df), dtype=np.float64)
        for fold_idx, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
            model.fit(X.iloc[train_idx], y[train_idx])
            pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
            oof_pred[valid_idx] = pred
            eval_rows.append({
                'model': model_name,
                'fold': fold_idx,
                'auc': float(roc_auc_score(y[valid_idx], pred)),
                'logloss': float(log_loss(y[valid_idx], pred)),
                'acc': float(accuracy_score(y[valid_idx], (pred >= 0.5).astype(int))),
            })

        full_auc = float(roc_auc_score(y, oof_pred))
        eval_rows.append({
            'model': model_name,
            'fold': 'oof',
            'auc': full_auc,
            'logloss': float(log_loss(y, oof_pred)),
            'acc': float(accuracy_score(y, (oof_pred >= 0.5).astype(int))),
        })
        if full_auc < min_auc_for_importance:
            continue

        model.fit(X, y)
        clf = model.named_steps['clf']
        if hasattr(clf, 'coef_'):
            raw_importance = np.abs(clf.coef_[0])
        elif hasattr(clf, 'feature_importances_'):
            raw_importance = clf.feature_importances_
        else:
            continue

        imp_df = pd.DataFrame({'model': model_name, 'feature': feature_cols, 'importance': raw_importance})
        imp_df['rank'] = imp_df['importance'].rank(method='dense', ascending=False).astype(int)
        importance_rows.extend(imp_df.sort_values('importance', ascending=False).head(50).to_dict('records'))

    return pd.DataFrame(eval_rows), pd.DataFrame(importance_rows)


def build_interaction_candidates(df: pd.DataFrame, feature_cols: list[str], top_k: int, max_pairs: int) -> pd.DataFrame:
    work_df = df[df['target'].notna()].copy().reset_index(drop=True)
    X = work_df[feature_cols].apply(pd.to_numeric, errors='coerce')
    y = work_df['target'].astype(int).to_numpy()
    X_imp = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=feature_cols)
    mi = mutual_info_classif(X_imp, y, random_state=42)
    top_features = pd.Series(mi, index=feature_cols).sort_values(ascending=False).head(top_k).index.tolist()

    rows = []
    for i, feat_a in enumerate(top_features):
        for feat_b in top_features[i + 1:]:
            a = X_imp[feat_a].to_numpy()
            b = X_imp[feat_b].to_numpy()
            prod = a * b
            ratio = a / (np.abs(b) + 1e-6)
            rows.append({'feature_a': feat_a, 'feature_b': feat_b, 'interaction': 'product', 'mi': float(mutual_info_classif(prod.reshape(-1, 1), y, random_state=42)[0])})
            rows.append({'feature_a': feat_a, 'feature_b': feat_b, 'interaction': 'ratio', 'mi': float(mutual_info_classif(ratio.reshape(-1, 1), y, random_state=42)[0])})
    return pd.DataFrame(rows).sort_values('mi', ascending=False).head(max_pairs).reset_index(drop=True)


def build_latent_feature_summary(df: pd.DataFrame, feature_cols: list[str], n_components: int, n_clusters: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from sklearn.cluster import KMeans

    work_df = df[df['target'].notna()].copy().reset_index(drop=True)
    X = work_df[feature_cols].apply(pd.to_numeric, errors='coerce')
    X_imp = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=feature_cols)
    X_scaled = StandardScaler().fit_transform(X_imp)

    pca = PCA(n_components=min(n_components, X_scaled.shape[1]), random_state=42)
    coords = pca.fit_transform(X_scaled)

    comp_rows = []
    for idx, comp in enumerate(pca.components_, start=1):
        loading = pd.Series(comp, index=feature_cols).sort_values(key=lambda s: s.abs(), ascending=False).head(10)
        for feature, value in loading.items():
            comp_rows.append({
                'component': f'pc{idx}',
                'feature': feature,
                'loading': float(value),
                'abs_loading': float(abs(value)),
                'explained_variance_ratio': float(pca.explained_variance_ratio_[idx - 1]),
            })

    coord_df = work_df[['split', 'sample_id', 'label', 'target']].copy()
    for idx in range(coords.shape[1]):
        coord_df[f'pc{idx + 1}'] = coords[:, idx]
    coord_df['cluster_id'] = KMeans(n_clusters=n_clusters, random_state=42, n_init=20).fit_predict(coords[:, : min(4, coords.shape[1])])
    if 'pc2' not in coord_df.columns:
        coord_df['pc2'] = 0.0

    cluster_df = (
        coord_df.groupby('cluster_id')
        .agg(
            n_samples=('sample_id', 'count'),
            unstable_rate=('target', 'mean'),
            pc1_mean=('pc1', 'mean'),
            pc2_mean=('pc2', 'mean'),
        )
        .reset_index()
    )
    cluster_df['nmi_vs_class'] = float(normalized_mutual_info_score(coord_df['target'], coord_df['cluster_id']))
    return pd.DataFrame(comp_rows), coord_df, cluster_df


In [6]:
catalog_df = build_labeled_catalog(DATA_ROOT, INCLUDE_TEST_FEATURE_EXPORT, MAX_SAMPLES_PER_SPLIT)
image_feat_df = build_image_feature_df(catalog_df)
geometry_df = build_geometry_df(DATA_ROOT, catalog_df)
motion_df = load_motion_features(DATA_ROOT, MOTION_CSV if MOTION_CSV.exists() else None)
analysis_df = build_analysis_df(image_feat_df, geometry_df, motion_df)

labeled_df = analysis_df[analysis_df['target'].notna()].copy().reset_index(drop=True)
feature_cols = get_feature_cols(labeled_df)

corr_matrix_df, target_corr_df = compute_correlation_outputs(labeled_df, feature_cols)
eval_df, importance_df = cross_validated_classification(labeled_df, feature_cols, MIN_AUC_FOR_IMPORTANCE)
interaction_df = build_interaction_candidates(labeled_df, feature_cols, TOP_INTERACTION_FEATURES, MAX_INTERACTIONS)
pca_loading_df, pca_coord_df, cluster_df = build_latent_feature_summary(labeled_df, feature_cols, PCA_COMPONENTS, CLUSTER_COUNT)

image_feat_df.to_csv(OUT_DIR / 'image_view_features.csv', index=False, encoding='utf-8-sig')
geometry_df.to_csv(OUT_DIR / 'geometry_sample_features.csv', index=False, encoding='utf-8-sig')
analysis_df.to_csv(OUT_DIR / 'class_analysis_features.csv', index=False, encoding='utf-8-sig')
corr_matrix_df.to_csv(OUT_DIR / 'feature_correlation_matrix.csv', encoding='utf-8-sig')
target_corr_df.to_csv(OUT_DIR / 'feature_target_correlation.csv', encoding='utf-8-sig')
eval_df.to_csv(OUT_DIR / 'classification_eval.csv', index=False, encoding='utf-8-sig')
importance_df.to_csv(OUT_DIR / 'classification_feature_importance.csv', index=False, encoding='utf-8-sig')
interaction_df.to_csv(OUT_DIR / 'interaction_candidates.csv', index=False, encoding='utf-8-sig')
pca_loading_df.to_csv(OUT_DIR / 'pca_top_loadings.csv', index=False, encoding='utf-8-sig')
pca_coord_df.to_csv(OUT_DIR / 'pca_sample_projection.csv', index=False, encoding='utf-8-sig')
cluster_df.to_csv(OUT_DIR / 'cluster_class_summary.csv', index=False, encoding='utf-8-sig')

print('saved:', OUT_DIR)
print('labeled samples:', len(labeled_df))
print('feature count:', len(feature_cols))
display(target_corr_df.head(20))
display(eval_df[eval_df['fold'].eq('oof')].sort_values(['auc', 'logloss'], ascending=[False, True]))
display(importance_df.sort_values(['model', 'importance'], ascending=[True, False]).groupby('model').head(15))
display(interaction_df.head(20))
display(cluster_df)


saved: /media/hdd0/whyz/structure-stability/outputs/physics_feature_analysis
labeled samples: 1100
feature count: 178


,corr_with_target,abs_corr_with_target
front_shadow_angle_deg,-0.500609,0.500609
abs_delta_std_b,-0.495020,0.495020
delta_std_b,-0.488818,0.488818
abs_delta_brightness_std,-0.482358,0.482358
delta_brightness_std,-0.476045,0.476045
delta_structure_area_ratio,-0.450824,0.450824
top_fill_ratio,-0.444045,0.444045
abs_delta_hue_entropy,-0.441286,0.441286
abs_delta_structure_area_ratio,-0.440584,0.440584
delta_hue_entropy,-0.439411,0.439411


,model,fold,auc,logloss,acc
11,rf,oof,0.997220,0.076580,0.967273
17,extra_trees,oof,0.996760,0.094089,0.962727
5,logreg,oof,0.970892,0.228161,0.964545


,model,feature,importance,rank
100,extra_trees,top_fill_ratio,0.036883,1
101,extra_trees,front_shadow_angle_deg,0.035841,2
102,extra_trees,front_structure_center_x,0.030969,3
103,extra_trees,abs_delta_structure_center_x,0.029633,4
104,extra_trees,mean_structure_center_x,0.026870,5
105,extra_trees,top_structure_center_x,0.020958,6
106,extra_trees,delta_structure_center_x,0.019183,7
107,extra_trees,top_centroid_dx,0.018665,8
108,extra_trees,mean_structure_bbox_w,0.018609,9
109,extra_trees,top_structure_bbox_w,0.016131,10


,feature_a,feature_b,interaction,mi
0,top_area_frac,top_structure_area_ratio,ratio,0.576883
1,top_centroid_dx,top_support_width_frac,ratio,0.575055
2,top_area_frac,collapse_margin_proxy,ratio,0.563024
3,top_centroid_dx,mean_structure_center_x,product,0.560827
4,top_structure_center_x,top_support_width_frac,product,0.559987
5,top_centroid_dx,mean_structure_center_x,ratio,0.559777
6,top_centroid_dx,top_structure_center_x,ratio,0.558236
7,top_structure_center_x,top_support_width_frac,ratio,0.556492
8,top_centroid_dx,top_structure_center_x,product,0.555218
9,front_structure_center_x,top_centroid_dx,product,0.554845


,cluster_id,n_samples,unstable_rate,pc1_mean,pc2_mean,nmi_vs_class
0,0,300,0.583333,-4.755145,-6.746172,0.055333
1,1,362,0.657459,3.864089,2.854716,0.055333
2,2,86,0.500000,13.014648,-6.929489,0.055333
3,3,352,0.272727,-3.100899,4.506763,0.055333
